# 03 — Pessoa e aplicação na autorização

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a seleção é declarada no deploy. A jornada `single-account` inclui 00–09 e 15–16; Registry, Terraform e RAG são exigidos somente pelas aulas que os utilizam. Ampliar a seleção exige atualizar a implantação pelo IaC. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 03](../docs/diagramas/svg/03_identity_agente_mcp.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "03")
print({"lab": "03", "dependencies": list(dependencies), "region": config["region"]})

### Conectar ao ambiente implantado

Lemos os outputs exatos do CloudFormation. O cliente não cria infraestrutura e não substitui um serviço indisponível.

In [ ]:
lab = AWSWorkshop(CONFIG, domain="platform")
print(lab.preflight())

### Obter dois tokens reais para a mesma pessoa

Os clientes confidenciais supervisor e riscos usam segredos distintos gerados pelo Cognito e guardados no Secrets Manager. Mudamos a aplicação que solicita o token, preservando o usuário.

In [ ]:
supervisor = lab.login("c", "supervisor")
riscos = lab.login("c", "riscos")
first, second = lab.claims(supervisor), lab.claims(riscos)
assert first["sub"] == second["sub"]
assert first["aud"] != second["aud"]
print({"same_person": True, "different_applications": True})

### Comparar decisões para a ferramenta Fraude

O supervisor recebe permit; a aplicação de Riscos deve receber uma negativa de Policy. Uma falha de rede não conta como negativa.

In [ ]:
arguments = {"cliente_id": "C001", "max_registros": 1}
print(await lab.call(supervisor, "especialista___consultar_fraude", arguments))
denied = await lab.expect_denial(riscos, "especialista___consultar_fraude", arguments)
print(denied)

### Testar um caminho alternativo

A restrição também cobre a ferramenta genérica de tabelas. A fonte revalida pessoa e aplicação, além da decisão do Gateway.

In [ ]:
print(await lab.expect_denial(riscos, "especialista___consultar_tabela",
                              {"tabela": "fraude"}))

**Entra ID:** faça a mesma comparação com tokens obtidos por dois clientes registrados. Valide `azp`/`appid` conforme a versão do token; não altere claims em um JWT já assinado.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.